# Ramp-state bucket capacity triage

Retrospective source/action support audit, with no return-label analysis.

In [1]:
import hashlib,json,os
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import numpy as np,pandas as pd,qlib
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd();card=root/'research/stage1_ramp_bucket_capacity_triage_20261112';plan=json.loads((card/'plan.json').read_text())
qlib.init(provider_uri=str(root/'data/qlib/stage1_event_guard_fullminute_20261028'),region='us',kernels=1)
identities={}
def source(role):
    path=root/plan['source_cards'][role];summary=json.loads((path/'summary.json').read_text())
    h=hashlib.sha256((path/'verification_manifest.json').read_bytes()).hexdigest()
    objects,receipt=load_verified_recorder_objects(path,expected_recorder_id=summary['recorder_id'],artifacts=[plan['source_artifacts'][role]],expected_manifest_sha256=h)
    identities[role]={'card':plan['source_cards'][role],'recorder_id':summary['recorder_id'],'manifest_sha256':h,'verified':True}
    return objects[plan['source_artifacts'][role]]
parent=source('parent')['authority'][['trade_date','risk_state']].drop_duplicates('trade_date')
bucket=source('bucket');action=source('action')
keys=['trade_date','bar_end_at','decision_at','fill_at']
assert not bucket.duplicated(keys).any() and not action.duplicated(keys).any()
# The serialized action source also contains outcome columns; no outcome column is accessed.
x=bucket[keys+['available','bucket_signed_net']].merge(action[keys+['decision_soxl_weight','decision_soxx_weight','decision_cash_weight','current_beta_cap','funded_change']],on=keys,how='inner',validate='one_to_one').merge(parent,on='trade_date',how='left',validate='many_to_one')
assert x.risk_state.notna().all()
x['day']=pd.to_datetime(x.trade_date)
slack=(x.current_beta_cap-3*x.decision_soxl_weight-x.decision_soxx_weight).clip(lower=0)
x['capacity']=np.minimum.reduce([np.full(len(x),.1),np.maximum(x.decision_cash_weight.to_numpy()-.15,0),slack.to_numpy(),np.maximum(1-x.decision_soxx_weight.to_numpy(),0)])
x['sell_pressure']=(-x.bucket_signed_net).clip(lower=0)
annual={}
for year in (2018,2019,2020,2021,2022):
    y=x.loc[x.day.dt.year.eq(year)]
    cell=y.loc[y.risk_state.eq(2)&y.available&y.capacity.gt(1e-10)]
    daymax=cell.groupby('day').capacity.max()
    annual[str(year)]={'parent_state2_days':int(parent.loc[pd.to_datetime(parent.trade_date).dt.year.eq(year),'risk_state'].eq(2).sum()),'source_capacity_days':int(cell.day.nunique()),'funded_days':int(cell.loc[cell.funded_change,'day'].nunique()),'positive_sell_pressure_days':int(cell.loc[cell.sell_pressure.gt(0),'day'].nunique()),'median_day_max_capacity_pct':None if daymax.empty else float(100*daymax.median())}
report={'experiment':plan['experiment'],'status':'not_admitted_sparse_ramp_bucket_cash_action_support','historical_research_passed':False,'source_identities':identities,'joined_intentions':len(x),'annual':annual,'new_model_fits':0,'new_native_accounts':0,'target_values_analyzed':False,'target_bytes_deserialized_with_saved_action_object':True,'limitations':['Retrospective source/capacity counts are not a frozen alpha test','Risk state and legal capacity are predecision but account-endogenous','funded_change is a postfill audit outcome and never a live feature/filter','Source bar first live receipt remains unqualified']}
(card/'summary.json').write_text(json.dumps(report,ensure_ascii=False,indent=2,allow_nan=False)+'\n')
print(json.dumps({'status':report['status'],'annual':annual},indent=2))

{
  "status": "not_admitted_sparse_ramp_bucket_cash_action_support",
  "annual": {
    "2018": {
      "parent_state2_days": 3,
      "source_capacity_days": 3,
      "funded_days": 2,
      "positive_sell_pressure_days": 1,
      "median_day_max_capacity_pct": 0.17466773686675707
    },
    "2019": {
      "parent_state2_days": 27,
      "source_capacity_days": 25,
      "funded_days": 22,
      "positive_sell_pressure_days": 19,
      "median_day_max_capacity_pct": 0.42904790853727226
    },
    "2020": {
      "parent_state2_days": 30,
      "source_capacity_days": 14,
      "funded_days": 10,
      "positive_sell_pressure_days": 10,
      "median_day_max_capacity_pct": 0.3361748604204684
    },
    "2021": {
      "parent_state2_days": 20,
      "source_capacity_days": 19,
      "funded_days": 13,
      "positive_sell_pressure_days": 17,
      "median_day_max_capacity_pct": 0.2855510877936118
    },
    "2022": {
      "parent_state2_days": 96,
      "source_capacity_days": 88,
   

## Conclusion

**No SOXX pass.** In exact f696+b472 ramp state with bucket source available and positive legal cash-to-SOXX capacity, 2019 has 25 independent days/22 funded days and 2020 has only 14/10. The median per-day maximum capacity is 0.429%/0.336% NAV. This is a retrospective feasibility triage, not a registered predictive gate. The specific ramp-only interaction is not fitted or sent to an account; see `conclusion.md`.